# 02 — Interactive Agent Tester & Quick Test
Test the vLLM endpoint live, try different game prompts, and compare output quality.

**Use this notebook to:**
- ✅ Sanity-check the vLLM server is working
- ✅ Test Thai / English language output
- ✅ Generate questions from any text snippet (no PDF needed)
- ✅ Compare question styles across game types
- ✅ Visualise question length & answer distribution from saved datasets

In [3]:
import json, os, re
from pathlib import Path
from openai import OpenAI

BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:8001/v1")
API_KEY  = os.getenv("AI_API_KEY",  "none")
MODEL    = os.getenv("AI_MODEL",    "Qwen/Qwen2.5-7B-Instruct-AWQ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# ── Quick sanity check ────────────────────────────────────────────────────────
try:
    models = client.models.list()
    name = models.data[0].id if models.data else "unknown"
    print(f"✅ vLLM ready  |  model: {name}")
    print(f"   Base URL: {BASE_URL}")
except Exception as e:
    print(f"❌ Cannot connect to vLLM: {e}")
    print()
    print("Steps to start vLLM:")
    print("  1. ssh aip04@br1.paas.ku.ac.th")
    print("  2. module load slurm")
    print("  3. sbatch ~/ku_prep_arena/ai/scripts/slurm_teacher.sh")
    print("  4. squeue -u aip04   (wait for RUNNING)")
    print("  5. Look at log for node name: tail -f ~/ku_prep_arena/teacher-<JOB>.log")
    print("  6. On LOCAL: ssh -N -L 8001:dgx-XX:8001 aip04@br1.paas.ku.ac.th")
    raise

✅ vLLM ready  |  model: Qwen/Qwen2.5-7B-Instruct-AWQ
   Base URL: http://localhost:8001/v1


## Game-Type Agent Comparison
สร้างคำถามจาก text เดียวกันด้วย prompt แต่ละ game type แล้วเปรียบเทียบผล

In [ ]:
import re, random

NO_CHINESE = (
    "CRITICAL: Do NOT output Chinese (中文), Japanese, Korean, Arabic, or any script "
    "not present in the document. Write ONLY in the same language as the document "
    "(Thai = ภาษาไทย, English = English)."
)

GAME_PROMPTS = {
    "flappy":  f"Keep questions SHORT (under 12 words) and choices very brief (1-4 words). Player reads while flying. {NO_CHINESE}",
    "racer":   f"Keep EVERY question under 12 words. Choices 1-4 words each — no long phrases. {NO_CHINESE}",
    "shooter": f"Focus on IDENTIFICATION questions: 'Which term describes…?'. Include plausible distractors. {NO_CHINESE}",
    "snake":   f"Prefer SEQUENTIAL questions: 'What is the FIRST step in…?'. Choices = different stages. {NO_CHINESE}",
    "bricks":  f"Focus on DEFINITIONS: 'What does X mean?'. Test precise vocabulary. {NO_CHINESE}",
}

def generate_for_game(text: str, game_type: str, n: int = 3) -> list[dict]:
    """Generate n questions for a specific game type. Returns list of dicts."""
    system = GAME_PROMPTS[game_type] + "\nRespond ONLY with a JSON array. No markdown, no explanation."
    user = (
        f"Create exactly {n} multiple-choice questions from the text below.\n"
        f"CRITICAL: Same language as the text. No Chinese.\n"
        f'Return ONLY: [{{"id":1,"question":"...","choices":["A","B","C","D"],"correct":0,"explanation":"..."}}]\n\n'
        f"Text:\n{text}"
    )
    res = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0.7,
        max_tokens=800,
    )
    raw = res.choices[0].message.content or "[]"
    # Bracket-balanced extraction
    start = raw.find('[')
    if start == -1:
        return []
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(raw[start:], start):
        if esc: esc = False; continue
        if ch == '\\' and in_str: esc = True; continue
        if ch == '"': in_str = not in_str
        elif not in_str:
            if ch == '[': depth += 1
            elif ch == ']':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(raw[start:i+1])
                    except json.JSONDecodeError:
                        return []
    return []

# ── ใส่ text ที่ต้องการทดสอบตรงนี้ ───────────────────────────────────────────
SAMPLE_TEXT = """
กระบวนการสังเคราะห์แสงแบ่งออกเป็น 2 ขั้นตอนหลัก คือ ปฏิกิริยาที่ต้องการแสง (Light reactions)
ซึ่งเกิดขึ้นที่เยื่อไทลาคอยด์ในคลอโรพลาสต์ และวัฏจักรคัลวิน (Calvin cycle)
ซึ่งเกิดขึ้นในสโตรมา กระบวนการนี้ใช้พลังงานแสงแปลงคาร์บอนไดออกไซด์และน้ำ
ให้เป็นกลูโคสและออกซิเจน
"""

N_QUESTIONS = 2   # จำนวนข้อต่อ game type

print(f"Generating {N_QUESTIONS} questions × {len(GAME_PROMPTS)} game types...\n")
all_results = {}
for game_type in GAME_PROMPTS:
    qs = generate_for_game(SAMPLE_TEXT, game_type, n=N_QUESTIONS)
    all_results[game_type] = qs
    status = f"{len(qs)} questions" if qs else "FAILED"
    print(f"  [{game_type:8s}] {status}")

print("\nDone.")

## Results — side-by-side comparison

In [ ]:
ABCD = ["A", "B", "C", "D"]
ICONS = {"flappy": "🐦", "racer": "🏎️", "shooter": "🚀", "snake": "🐍", "bricks": "🧱"}

for game_type, qs in all_results.items():
    icon = ICONS.get(game_type, "")
    print(f"\n{'='*60}")
    print(f"{icon}  {game_type.upper()}  ({len(qs)} questions)")
    print("="*60)
    if not qs:
        print("  (no questions generated)")
        continue
    for q in qs:
        q_len = len(q.get("question","").split())
        c_lens = [len(c.split()) for c in q.get("choices",[])]
        print(f"\n  Q: {q['question']}  [{q_len}w]")
        for i, c in enumerate(q.get("choices", [])[:4]):
            mark = ">" if i == q.get("correct") else " "
            print(f"    {mark}{ABCD[i]}) {c}  [{c_lens[i] if i < len(c_lens) else '?'}w]")
        print(f"    => {q.get('explanation','')[:80]}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

GENERATED = Path("../dataset/generated")
Path("../eval/results").mkdir(parents=True, exist_ok=True)

rows = []
for f in sorted(GENERATED.glob("*.json")):
    questions = json.loads(f.read_text(encoding="utf-8"))
    for q in questions:
        choices = [str(c) for c in q.get("choices", [])]  # cast to str — some may be float
        rows.append({
            "game_type":      q.get("game_type", f.stem.rsplit("_", 1)[-1]),
            "q_len":          len(q.get("question", "").split()),
            "choice_avg_len": sum(len(c.split()) for c in choices) / max(len(choices), 1),
            "correct":        q.get("correct", 0),
            "difficulty":     q.get("difficulty", 2),
        })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} questions from notebook 01 | {df['game_type'].nunique()} game types")
print(f"Difficulty: {df['difficulty'].value_counts().sort_index().to_dict()}")
df.groupby("game_type").size()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Question word count
df.groupby("game_type")["q_len"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="#336600", alpha=0.85
)
axes[0].set_title("Avg Question Length (words)")
axes[0].set_xlabel("Words")

# Choice word count
df.groupby("game_type")["choice_avg_len"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#f5c518", alpha=0.85
)
axes[1].set_title("Avg Choice Length (words)")
axes[1].set_xlabel("Words")

plt.tight_layout()
plt.savefig("../eval/results/agent_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: eval/results/agent_comparison.png")

## Answer distribution per game type

In [ ]:
pivot = df.groupby(["game_type", "correct"]).size().unstack(fill_value=0)
# Map column indices to letters dynamically (handles out-of-range correct values)
labels = ["A", "B", "C", "D", "E", "F"]
pivot.columns = [labels[c] if isinstance(c, int) and c < len(labels) else str(c)
                 for c in pivot.columns]
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Show bad data if any columns beyond D exist
valid = [c for c in pivot.columns if c in ["A", "B", "C", "D"]]
invalid = [c for c in pivot.columns if c not in ["A", "B", "C", "D"]]
if invalid:
    bad = df[df["correct"] >= 4][["game_type", "correct"]].value_counts()
    print(f"WARNING: {df['correct'].ge(4).sum()} questions have correct >= 4 (invalid):")
    print(bad.to_string())
    print()

ax = pivot_pct[valid].plot(kind="bar", figsize=(10, 4), colormap="Set2", edgecolor="white")
ax.axhline(25, color="black", linestyle="--", linewidth=1, label="Ideal 25%")
ax.set_title("Answer Distribution by Game Type (%)")
ax.set_xlabel("")
ax.set_ylabel("Percentage (%)")
ax.legend(loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../eval/results/answer_distribution.png", bbox_inches="tight")
plt.show()